# LeapQuant KDA 教程 (1/5): KDA 的账本 —— 每一列自己遗忘

**KDA** (Kimi Delta Attention) 是 Kimi-Linear / Kimi K3 混合模型里线性注意力层的变体。
教程 1–5 里我们已经把 **GDN** (Qwen3-Next 用的 Gated DeltaNet) 的 LeapQuant 量化讲完了;
KDA 与 GDN 的递推**只差一行** —— 但这一行让"门"从每头一个数变成每头一个向量。

本系列 (KDA 篇 1–5) 沿着同样的阶梯走一遍:

| 册 | 主题 |
|---|---|
| 1 | KDA 状态与 per-K 衰减 (本册) |
| 2 | 门的来历: `A_log`/`dt_bias` 与两种展开式 + l2norm |
| 3 | KDA 状态的 int8 快照 (rank-r + 双尺度) |
| 4 | 窗口扫描: flush 间隔 vs 漂移 |
| 5 | 落地: `kimi_kda` 调用点与 triton 核 |

In [1]:
import sys, math
import torch

sys.path.insert(0, "/workspace/y00924007/vllm-ascend")  # 开发仓 (kernel / 量化器所在)
torch.npu.set_device(0)
DEV = torch.device("npu:0")
torch.manual_seed(7)
print("device:", torch.npu.get_device_name(0))

/usr/local/python3.11.10/lib/python3.11/site-packages/torch_npu/_init/core/optional_features.py:20: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. Do not set it to 1 to avoid unexpected errors.
  warnings.warn(


device: Ascend910_9382


## 1. 先复习 GDN: 整个账本一起衰减

线性注意力的状态 $S$ $[V, K]$ 是一本"账本" (教程 GDN-1 的比喻): 每 token 来一笔账
$\beta\, v k^{\top}$。GDN 每步先给**整本账**乘同一个衰减 $e^{g}$ ($g$ 是每头一个标量):

In [2]:
def gdn_step(S, q, k, v, g, beta):
    """S [V,K]; q,k [K]; v [V]; g 标量; beta 标量."""
    S = S * math.exp(g)                       # 整本账一起衰减
    sk = S @ k                                # 读回: 当前账里 k 欠多少
    v_tilde = (v - sk) * beta                 # 冲销旧账 + 记新账
    S = S + torch.outer(v_tilde, k)
    o = S @ q
    return S, o

K, V = 4, 4
S = torch.zeros(V, K, device=DEV)
k = torch.nn.functional.normalize(torch.randn(K, device=DEV), dim=-1)
v = torch.randn(V, device=DEV)
S, o = gdn_step(S, k, k, v, g=-0.1, beta=0.7)
print("GDN 一步后状态:\n", S)

[W1111 16:17:23.200582156 NPUCachingAllocator.cpp:204] Warning: The current CANN and HDK(driver) versions require processing for 32 padding size, with memory allocation. (function operator())


GDN 一步后状态:
 tensor([[-0.0783, -0.1470,  0.2165, -0.1720],
        [-0.0043, -0.0081,  0.0119, -0.0095],
        [-0.2170, -0.4071,  0.5997, -0.4763],
        [ 0.0907,  0.1702, -0.2507,  0.1991]], device='npu:0')


[W1111 16:17:24.586489229 ToKernelNpu.cpp:40] Warning: Device do not support double dtype now, dtype cast replace with float. (function operator())


## 2. KDA 的差别: 衰减是一个向量 $g \in \mathbb{R}^{K}$

KDA 把 $g$ 变成**每个键维度一个数**: 状态第 $j$ 列乘 $e^{g_j}$。

直觉: 账本的每个"键维度"有自己的记忆时长 —— 有的维度记长期依赖 ( $g_j \approx 0$ ),
有的快速遗忘 ( $g_j` 很负 )。这让 KDA 能**同时**做长记忆和快速更新, 是 Kimi-Linear
在长文本上表现好的关键设计之一。

In [3]:
def kda_step(S, q, k, v, g, beta):
    """S [V,K]; q,k [K]; v [V]; g [K] (每列一个衰减!); beta 标量."""
    S = S * torch.exp(g)[None, :]             # <-- 与 GDN 唯一的差别: 逐列衰减
    sk = S @ k
    v_tilde = (v - sk) * beta
    S = S + torch.outer(v_tilde, k)
    o = S @ q
    return S, o

K, V = 8, 8
g = torch.tensor([-0.01, -0.05, -0.2, -0.8, -2.0, -0.03, -0.1, -0.5], device=DEV)
S = torch.zeros(V, K, device=DEV)
for t in range(64):
    k = torch.nn.functional.normalize(torch.randn(K, device=DEV), dim=-1)
    v = torch.randn(V, device=DEV)
    S, _ = kda_step(S, k, k, v, g, beta=0.7)

col = S.norm(dim=0)                            # 每列的范数
print("g        :", [f"{x:5.2f}" for x in g.tolist()])
print("稳态列范数:", [f"{x:5.2f}" for x in col.tolist()])
print("\n--> 衰减越负的列, 稳态范数越小: 每列记忆时长确实由 g_j 决定")

g        : ['-0.01', '-0.05', '-0.20', '-0.80', '-2.00', '-0.03', '-0.10', '-0.50']
稳态列范数: [' 1.41', ' 2.12', ' 0.96', ' 1.08', ' 0.98', ' 1.09', ' 0.80', ' 0.58']

--> 衰减越负的列, 稳态范数越小: 每列记忆时长确实由 g_j 决定


## 3. 两种写法恒等 —— 为什么实现里用第二种

论文附录 A 给出的恒等式 ($\alpha = e^{g}$, 逐列):
$$(I - \beta k k^{\top})\,\mathrm{Diag}(\alpha)\,S + \beta k v^{\top}
\;\equiv\; \mathrm{Diag}(\alpha)\,S + \beta\, k\, (v - S^{\top}(\alpha \odot k))^{\top}$$

左边是"先衰减、再正交冲销"的几何写法; 右边是**实现**用的写法 —— 衰减之后一笔外积搞定,
不需要显式构造 $I - \beta kk^{\top}$。数值上验证一下 (教学规模):

In [4]:
V, K = 6, 5
S = torch.randn(V, K, device=DEV) * 0.3
kn = torch.nn.functional.normalize(torch.randn(K, device=DEV), dim=-1)  # 单位范数 k
v = torch.randn(V, device=DEV)
alpha = torch.exp(torch.randn(K, device=DEV) * 0.3 - 0.3)               # e^{g_j} ∈ (0,1)
beta = 0.6

S_alpha = S * alpha[None, :]                                   # Diag(α) S
lhs = S_alpha @ (torch.eye(K, device=DEV) - beta * torch.outer(kn, kn)) + beta * torch.outer(v, kn)
rhs = S_alpha + beta * torch.outer(v - S_alpha @ kn, kn)

print("lhs == rhs, max|diff| =", (lhs - rhs).abs().max().item())

lhs == rhs, max|diff| = 5.960464477539063e-08


## 4. 长期行为对比: 状态范数

- **GDN**: 整本账一个衰减 → 稳态范数由标量 $g$ 决定, 所有列同涨同跌;
- **KDA**: 每列自己的衰减 → 稳态范数**分层** (上一节已经看到), 状态的谱更不均匀
  —— 这会直接影响后面量化的误差分布 (教程 3/4 再展开)。

In [5]:
def run_norms(step_fn, T=256, seed=3):
    torch.manual_seed(seed)
    K, V = 16, 16
    g_scalar, beta = -0.3, 0.7                       # GDN 标量门
    g_vec = -0.3 * torch.ones(K, device=DEV)          # 故意取同一均值的向量门
    g_vec[: K // 2] -= 0.9                            # 一半维度衰减更狠
    S = torch.zeros(V, K, device=DEV)
    hist = []
    for t in range(T):
        k = torch.nn.functional.normalize(torch.randn(K, device=DEV), dim=-1)
        q = torch.nn.functional.normalize(torch.randn(K, device=DEV), dim=-1)
        v = torch.randn(V, device=DEV)
        S, _ = step_fn(S, q, k, v, g_vec if step_fn is kda_step else g_scalar, beta)
        hist.append(S.norm().item())
    return hist

h_gdn = run_norms(gdn_step)
h_kda = run_norms(kda_step)
print(f"GDN 稳态范数 (末 32 步均值): {sum(h_gdn[-32:]) / 32:.3f}")
print(f"KDA 稳态范数 (末 32 步均值): {sum(h_kda[-32:]) / 32:.3f}   <-- 同均值门, 更快的列拖低了总量")
print(f"GDN 前 64 步 vs 末 64 步: {sum(h_gdn[:64]) / 64:.1f} -> {sum(h_gdn[-64:]) / 64:.1f}")
print(f"KDA 前 64 步 vs 末 64 步: {sum(h_kda[:64]) / 64:.1f} -> {sum(h_kda[-64:]) / 64:.1f}")

GDN 稳态范数 (末 32 步均值): 4.040
KDA 稳态范数 (末 32 步均值): 3.517   <-- 同均值门, 更快的列拖低了总量
GDN 前 64 步 vs 末 64 步: 3.9 -> 4.1
KDA 前 64 步 vs 末 64 步: 3.4 -> 3.6


## 小结 (KDA 教程 1)

- KDA 与 GDN 的递推只差一处: 衰减 $e^{g}$ 从**每头标量**变成**每头每键维度一个数**;
- 两条代数写法恒等, 实现取 $\mathrm{Diag}(\alpha)S + \beta k(v - S^{\top}(\alpha \odot k))^{\top}$;
- 每列独立衰减 → 稳态列范数分层: 有的列记长期, 有的列快速遗忘;
- 状态形状仍是 $[V, K]$ —— LeapQuant 量化器作用于矩阵本身, 与门的结构无关 (教程 3 验证)。